<a href="https://colab.research.google.com/github/nikunj06/ingres/blob/main/ingres.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --ignore-installed blinker

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('rajasthan_groundwater_blocks_20250913_230226.csv')

# Find columns that are entirely empty (all NaN) OR all zeros
empty_cols = df.columns[df.isna().all()].tolist()
zero_cols = df.columns[(df == 0).all()].tolist()

# Remove duplicates (columns that are both empty and zero)
cols_to_drop = list(set(empty_cols + zero_cols))

print(f"Original shape: {df.shape}")
print(f"Empty columns ({len(empty_cols)}): {empty_cols}")
print(f"Zero columns ({len(zero_cols)}): {zero_cols}")
print(f"Total columns to drop ({len(cols_to_drop)}): {cols_to_drop}")
print(f"Cleaned shape: {df.drop(columns=cols_to_drop).shape}")

# Drop empty and zero columns, save cleaned file
df_clean = df.drop(columns=cols_to_drop)
df_clean.to_csv('cleaned_rajasthan_groundwater.csv', index=False)
print("Cleaned file saved as 'cleaned_rajasthan_groundwater.csv'")
print("\nCleaned data preview:")
print(df_clean.head())


Original shape: (369, 106)
Empty columns (0): []
Zero columns (58): ['recharge_worthy_hilly_area_ha', 'recharge_worthy_forest_area_ha', 'recharge_worthy_paved_area_ha', 'recharge_worthy_unpaved_area_ha', 'non_recharge_worthy_command_area_ha', 'non_recharge_worthy_non_command_area_ha', 'non_recharge_worthy_poor_quality_area_ha', 'non_recharge_worthy_hilly_area_ha', 'non_recharge_worthy_forest_area_ha', 'non_recharge_worthy_paved_area_ha', 'non_recharge_worthy_unpaved_area_ha', 'hilly_area_ha', 'forest_area_ha', 'paved_area_ha', 'unpaved_area_ha', 'transpiration_loss_ham', 'evaporation_loss_ham', 'et_loss_ham', 'static_gw_total_ham', 'static_gw_command_ham', 'static_gw_non_command_ham', 'static_gw_poor_quality_ham', 'additional_recharge_total_ham', 'additional_recharge_flood_prone_ham', 'additional_recharge_shallow_area_ham', 'additional_recharge_spring_discharge_ham', 'projected_domestic_utilization_ham', 'projected_total_utilization_ham', 'semi_confined_aquifer_total_ham', 'semi_confin

In [ ]:
import pandas as pd

class CSVRetrievalEvaluator:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path, sep='\t')
        self.prepare_text_corpus()

    def prepare_text_corpus(self):
        """Convert CSV rows to text chunks for retrieval"""
        text_chunks = []
        for idx, row in self.df.iterrows():
            # Combine all fields into searchable text
            chunk = ' '.join([
                f"{col}: {str(val)}"
                for col, val in row.items()
                if pd.notna(val)
            ])
            text_chunks.append(chunk)

        self.text_chunks = text_chunks

    def relational_search(self, query):
        """
        Simulate exact keyword matching (relational DB approach)
        For CSV: checks if ALL terms in the query appear in the row
        """
        results = []
        query_terms = query.lower().split()

        for idx, chunk in enumerate(self.text_chunks):
            chunk_lower = chunk.lower()
            # Check if ALL query terms exist in this chunk
            if all(term in chunk_lower for term in query_terms):
                results.append(idx)

        return results

    def relational_search_exact(self, query):
        """
        Alternative: strict exact phrase matching
        (Original behavior - keeps for comparison)
        """
        results = []
        query_lower = query.lower()

        for idx, chunk in enumerate(self.text_chunks):
            if query_lower in chunk.lower():
                results.append(idx)

        return results

    def evaluate_group_a(self):
        """Group A: Simple keywords/entity names"""
        print("\n=== GROUP A: SIMPLE KEYWORDS ===")

        # Define your test terms - modify these based on your data
        group_a_terms = [
            "AJMER",           # Exact district name
            "over_exploited",  # Category value
            "ALWAR"            # Another district name
        ]

        results = {}
        for term in group_a_terms:
            rel_results = self.relational_search(term)

            results[term] = {
                'relational_count': len(rel_results),
                'matching_indices': rel_results[:5]  # Show first 5 matches
            }

            print(f"\nTerm: '{term}'")
            print(f"  Relational DB found: {len(rel_results)} results")
            print(f"  Sample matching rows: {rel_results[:3]}")

        return results

# Usage
evaluator = CSVRetrievalEvaluator('/content/cleaned_rajasthan_groundwater.csv')
results = evaluator.evaluate_group_a()


=== GROUP A: SIMPLE KEYWORDS ===

Term: 'AJMER'
  Relational DB found: 12 results
  Sample matching rows: [0, 1, 2]

Term: 'over_exploited'
  Relational DB found: 369 results
  Sample matching rows: [0, 1, 2]

Term: 'ALWAR'
  Relational DB found: 18 results
  Sample matching rows: [12, 13, 14]


In [ ]:
import pandas as pd
import numpy as np


class CSVRetrievalEvaluator:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)  # Comma-separated
        self.prepare_text_corpus()

    def prepare_text_corpus(self):
        """Convert CSV rows to text chunks for retrieval"""
        text_chunks = []
        for idx, row in self.df.iterrows():
            chunk = ' '.join([
                f"{col}: {str(val)}"
                for col, val in row.items()
                if pd.notna(val)
            ])
            text_chunks.append(chunk)
        self.text_chunks = text_chunks

    def relational_search(self, query):
        """
        Simulate exact keyword matching (relational DB approach)
        For CSV: checks if ALL terms in the query appear in the row
        """
        results = []
        query_terms = query.lower().split()

        for idx, chunk in enumerate(self.text_chunks):
            chunk_lower = chunk.lower()
            # Check if ALL query terms exist in this chunk
            if all(term in chunk_lower for term in query_terms):
                results.append(idx)

        return results

    def generate_group_b_patterns(self, n_patterns=30):
        """
        Group B for CSV: Multi-column value combinations
        Tests if the system can retrieve rows matching specific attribute combinations
        """
        patterns = []

        # Sample diverse rows
        total_rows = len(self.df)
        sample_step = max(1, total_rows // (n_patterns + 1))
        sample_indices = list(range(0, min(total_rows, n_patterns * sample_step), sample_step))[:n_patterns]

        for idx in sample_indices:
            row = self.df.iloc[idx]

            # Pattern Type 1: Location + Category (3-4 fields)
            pattern = f"{row['district_name']} {row['block_name']} {row['total_category']}"
            if pattern not in patterns:
                patterns.append(pattern)
                continue

            # Pattern Type 2: Location + Key Metric (3-4 fields)
            if len(patterns) < n_patterns:
                pattern = f"{row['state_name']} {row['district_name']} {row['level']} {row['is_urban']}"
                if pattern not in patterns:
                    patterns.append(pattern)
                    continue

            # Pattern Type 3: Numeric range query (2-3 fields)
            if len(patterns) < n_patterns:
                pattern = f"{row['district_name']} {row['stage_of_development_percent']} {row['recharge_worthy_percentage']}"
                if pattern not in patterns:
                    patterns.append(pattern)

        return patterns[:n_patterns]

    def evaluate_group_b(self):
        """
        Group B: Multi-column value combination retrieval
        Tests: Can the system find rows matching specific combinations of field values?
        """
        print("\n" + "="*70)
        print("GROUP B: MULTI-COLUMN VALUE COMBINATIONS")
        print("="*70)

        patterns = self.generate_group_b_patterns(n_patterns=30)

        print(f"\nGenerated {len(patterns)} test patterns:\n")
        for i, pattern in enumerate(patterns, 1):
            print(f"B-{i}: {pattern}")

        print("\n" + "-"*70)
        print("RETRIEVAL RESULTS")
        print("-"*70)

        results = {}

        for i, pattern in enumerate(patterns, 1):
            # Count how many rows should match this pattern
            # (In real data, same combinations might appear multiple times)
            source_count = sum(1 for chunk in self.text_chunks
                              if all(term.lower() in chunk.lower()
                                    for term in pattern.split()))

            # Relational search (exact match)
            rel_results = self.relational_search(pattern)

            recall = len(rel_results) / source_count if source_count > 0 else 0

            results[f"B-{i}"] = {
                'pattern': pattern,
                'expected_matches': source_count,
                'relational_found': len(rel_results),
                'recall': recall
            }

            print(f"\nB-{i}: '{pattern}'")
            print(f"  Expected matches: {source_count}")
            print(f"  Relational DB found: {len(rel_results)}")
            print(f"  Recall: {recall:.1%}")

        return results
    def test_row_completion(self):
      """Row Completion Test: Does retrieval find complete row signatures?"""
      print("\n" + "="*80)
      print("ROW COMPLETION TEST: FULL ROW RETRIEVAL")
      print("="*80)

      # Test 5 representative rows
      test_rows = [1, 10, 50, 100, 200]  # Diverse indices
      results = {}

      for row_idx in test_rows:
          row = self.df.iloc[row_idx]

          # Generate partial query (3-4 fields)
          query_terms = []
          key_fields = ['district_name', 'block_name', 'level', 'total_category']
          for field in key_fields:
              if field in self.df.columns and pd.notna(row.get(field)):
                  query_terms.append(str(row[field]))

          if len(query_terms) < 3:
              continue

          query = ' '.join(query_terms[:4])

          # Retrieve
          retrieved_rows = self.relational_search(query)

          # Ground truth: This exact row
          complete_match_count = 0

          print(f"\nRow {row_idx}: Query='{query}'")
          print(f"  Expected row fields: {len(self.df.columns)}")

          for retrieved_idx in retrieved_rows[:5]:  # Top 5
              retrieved_row = self.df.iloc[retrieved_idx]

              # Check field overlap (completion score)
              common_fields = 0
              total_fields = 0

              for col in self.df.columns:
                  expected_val = row.get(col)
                  retrieved_val = retrieved_row.get(col)

                  if pd.notna(expected_val) and pd.notna(retrieved_val):
                      total_fields += 1
                      if str(expected_val) == str(retrieved_val):
                          common_fields += 1

              completion_score = common_fields / total_fields if total_fields > 0 else 0

              if retrieved_idx == row_idx:  # Exact row match
                  complete_match_count += 1
                  print(f"  ✅ EXACT ROW #{retrieved_idx} (completion: {completion_score:.0%})")
              else:
                  print(f"  {retrieved_idx}: {completion_score:.0%} field match")

          recall = 1 if complete_match_count > 0 else 0
          results[row_idx] = {
              'query': query,
              'retrieved_count': len(retrieved_rows),
              'exact_row_found': complete_match_count > 0,
              'recall': recall
          }

      overall_recall = sum(r['recall'] for r in results.values()) / len(results)
      print(f"\n{'='*80}")
      print(f"ROW COMPLETION SUMMARY")
      print(f"{'='*80}")
      print(f"Exact row recall: {overall_recall:.0%} ({sum(r['recall'] for r in results.values())}/{len(results)})")

      return results


# Usage
if __name__ == "__main__":
    evaluator = CSVRetrievalEvaluator('/content/cleaned_rajasthan_groundwater.csv')
    #results = evaluator.evaluate_group_b()
    # Usage
    evaluator.test_row_completion()


    # Save results
    import os
    results_df = pd.DataFrame(results).T
    results_df.to_csv('group_b_results.csv', index=False)
    print(f"\nResults saved to 'group_b_results.csv'")


ROW COMPLETION TEST: FULL ROW RETRIEVAL

Row 1: Query='AJMER AJMER_URBAN BLOCK over_exploited'
  Expected row fields: 48
  ✅ EXACT ROW #1 (completion: 100%)

Row 10: Query='AJMER SHRINAGAR_RURAL BLOCK over_exploited'
  Expected row fields: 48
  ✅ EXACT ROW #10 (completion: 100%)

Row 50: Query='BARMER DISTRICT over_exploited'
  Expected row fields: 48
  ✅ EXACT ROW #50 (completion: 100%)
  51: 44% field match
  52: 44% field match
  53: 44% field match
  54: 40% field match

Row 100: Query='BIKANER KOLAYAT BLOCK safe'
  Expected row fields: 48
  ✅ EXACT ROW #100 (completion: 100%)

Row 200: Query='JAISALMER JAISALMER_RURAL BLOCK over_exploited'
  Expected row fields: 48
  ✅ EXACT ROW #200 (completion: 100%)

ROW COMPLETION SUMMARY
Exact row recall: 100% (5/5)

Results saved to 'group_b_results.csv'


In [ ]:
!pip install faiss-cpu elasticsearch pyngrok groq streamlit sentence-transformers pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.6/963.6 kB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 149.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 152.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


In [ ]:
%%writefile data_loader.py
import json
import pandas as pd
from elasticsearch import Elasticsearch, helpers
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict, Any, Optional

ES_CLOUD_ID = "sih:dXMtY2VudHJhbDEuZ2NwLmNsb3VkLmVzLmlvOjQ0MyQ2ZDYxNDYxMThjNGM0MGEyOWU2M2RkNDRiYWQ2MjlmNiQyNGM2YzcyNDY2NjY0N2M2OTA2MmQzODRkOTc0ZDE1ZQ=="
ES_USERNAME = "elastic"
ES_PASSWORD = "geERTNU9q0omxJTb0NmAswkJ"

class ElasticGroundwaterSearch:
    def __init__(self, json_file: str, index_name: str = "groundwater", embedding_model: str = "all-MiniLM-L6-v2"):
        self.json_file = json_file
        self.index_name = index_name
        self.es = Elasticsearch(
            cloud_id=ES_CLOUD_ID,
            basic_auth=(ES_USERNAME, ES_PASSWORD)
        )
        self.model = SentenceTransformer(embedding_model)
        # Define invalid block names that should be excluded
        self.invalid_blocks = {
            "total", "district_total", "state_total", "",
            "TOTAL", "Total", "District Total", "STATE_TOTAL",
            "district total", "state total", "grand total",
            "GRAND_TOTAL", "DISTRICT_TOTAL"
        }
        self._create_index()
        self._load_data()

    def _create_index(self):
        if self.es.indices.exists(index=self.index_name):
            return

        index_config = {
            "mappings": {
                "properties": {
                    "state_name": {"type": "keyword"},
                    "district_name": {"type": "keyword"},
                    "block_name": {"type": "keyword"},
                    "total_gw_availability_ham": {"type": "float"},
                    "net_gw_availability_ham": {"type": "float"},
                    "stage_of_development_percent": {"type": "float"},
                    "rainfall_total_mm": {"type": "float"},
                    "total_category": {"type": "keyword"},
                    "combined_text": {"type": "text"},
                    "embedding": {"type": "dense_vector", "dims": 384, "index": True, "similarity": "cosine"},
                    "unique_id": {"type": "keyword"},
                    "is_individual_block": {"type": "boolean"}  # Flag to identify individual blocks
                }
            }
        }
        self.es.indices.create(index=self.index_name, body=index_config)
        print(f"✅ Created index: {self.index_name}")

    def _is_valid_block(self, record: Dict[str, Any]) -> bool:
        """
        Check if a record represents an individual block (not a summary/total)
        """
        block_name = str(record.get('block_name', '')).strip()
        district_name = str(record.get('district_name', '')).strip()
        state_name = str(record.get('state_name', '')).strip()

        # Skip if block name is in invalid list
        if block_name.lower() in [name.lower() for name in self.invalid_blocks]:
            return False

        # Skip if block name is empty or None
        if not block_name or block_name.lower() == 'none' or block_name.lower() == 'null':
            return False

        # Skip if block name is same as district name (likely a district total)
        if block_name.lower() == district_name.lower():
            return False

        # Skip if it's a state-level record (state = district)
        if state_name.lower() == district_name.lower():
            return False

        return True

    def _load_data(self):
        with open(self.json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        actions = []
        seen_records = set()
        valid_blocks = 0
        skipped_blocks = 0

        for rec in data:
            # Check if this is a valid individual block
            if not self._is_valid_block(rec):
                skipped_blocks += 1
                block_name = rec.get('block_name', 'EMPTY')
                district_name = rec.get('district_name', 'UNKNOWN')
                print(f"⚠️ Skipping summary/invalid record: {district_name} > {block_name}")
                continue

            # Create unique identifier
            unique_key = f"{rec.get('state_name', '')}_{rec.get('district_name', '')}_{rec.get('block_name', '')}"

            # Skip if we've already seen this combination
            if unique_key in seen_records:
                print(f"⚠️ Skipping duplicate: {unique_key}")
                skipped_blocks += 1
                continue
            seen_records.add(unique_key)

            # Add flag to identify this as an individual block
            rec["is_individual_block"] = True

            # Build combined text with better formatting
            parts = [
                f"State: {rec.get('state_name', 'N/A')}",
                f"District: {rec.get('district_name', 'N/A')}",
                f"Block: {rec.get('block_name', 'N/A')}",
                f"Total GW Availability: {rec.get('total_gw_availability_ham', 'N/A')} HAM",
                f"Net GW Availability: {rec.get('net_gw_availability_ham', 'N/A')} HAM",
                f"Development Stage: {rec.get('stage_of_development_percent', 'N/A')}%",
                f"Rainfall: {rec.get('rainfall_total_mm', 'N/A')} mm",
                f"Category: {rec.get('total_category', 'N/A')}"
            ]
            rec["combined_text"] = " | ".join(parts)
            rec["unique_id"] = unique_key

            # Generate embedding
            emb = self.model.encode(rec["combined_text"], convert_to_numpy=True, normalize_embeddings=True)
            rec["embedding"] = emb.tolist()

            actions.append({
                "_index": self.index_name,
                "_id": unique_key,
                "_source": rec
            })
            valid_blocks += 1

        helpers.bulk(self.es, actions)
        print(f"✅ Indexed {valid_blocks} valid individual blocks")
        print(f"⚠️ Skipped {skipped_blocks} summary/invalid records")

    def _get_base_individual_blocks_filter(self):
        """
        Base filter to ensure we only get individual blocks in searches
        """
        return {
            "bool": {
                "must": [
                    {"term": {"is_individual_block": True}}
                ],
                "must_not": [
                    {"terms": {"block_name": list(self.invalid_blocks)}},
                    {"term": {"block_name": ""}},
                ]
            }
        }

    def search(self, query: str, top_k: int = 10, min_score: float = 0.1, district_name: str = None):
        """
        Improved hybrid search with better scoring and deduplication - ONLY individual blocks

        Args:
            query: Search query string
            top_k: Number of results to return
            min_score: Minimum relevance score threshold
            district_name: Optional district filter for targeted search
        """
        query_vector = self.model.encode([query], convert_to_numpy=True)[0].tolist()

        # Build must clauses
        must_clauses = [self._get_base_individual_blocks_filter()]

        # Add district filter if provided
        if district_name:
            must_clauses.append({"term": {"district_name.keyword": district_name.upper()}})

        body = {
            "size": top_k * 2,
            "query": {
                "bool": {
                    "must": must_clauses,
                    "should": [
                        {
                            "multi_match": {
                                "query": query,
                                "fields": [
                                    "state_name^3",
                                    "district_name^3",
                                    "block_name^2",
                                    "total_category^2",
                                    "combined_text^1"
                                ],
                                "type": "best_fields",
                                "boost": 2.5  # Emphasize text matching more
                            }
                        },
                        {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    # Improved vector similarity scoring with better normalization
                                    "source": """
                                        double similarity = cosineSimilarity(params.query_vector, 'embedding');
                                        return Math.max(0, (similarity + 1.0) / 2.0 * 5.0);
                                    """,
                                    "params": {"query_vector": query_vector}
                                },
                                "boost": 1.5  # Less emphasis on semantic similarity for now
                            }
                        }
                    ],
                    "minimum_should_match": 1
                }
            },
            "min_score": min_score,
            "_source": {
                "excludes": ["embedding"]
            }
        }

        res = self.es.search(index=self.index_name, body=body)
        hits = res["hits"]["hits"]

        # Post-process results
        unique_results = []
        seen_locations = set()

        for hit in hits:
            source = hit["_source"]
            location_key = f"{source.get('state_name', '')}_{source.get('district_name', '')}_{source.get('block_name', '')}"

            if location_key in seen_locations:
                continue
            seen_locations.add(location_key)

            source["_relevance_score"] = hit["_score"]
            unique_results.append(source)

            if len(unique_results) >= top_k:
                break

        return unique_results

    def detect_district_in_query(self, query: str) -> str:
        """
        Extract district name from query using comprehensive district list

        Returns:
            District name in uppercase if found, None otherwise
        """
        query_lower = query.lower()

        # Comprehensive list of Rajasthan districts (including common variations)
        rajasthan_districts = {
            'ajmer': 'AJMER',
            'jaipur': 'JAIPUR',
            'jodhpur': 'JODHPUR',
            'udaipur': 'UDAIPUR',
            'kota': 'KOTA',
            'bikaner': 'BIKANER',
            'alwar': 'ALWAR',
            'bharatpur': 'BHARATPUR',
            'pali': 'PALI',
            'sikar': 'SIKAR',
            'churu': 'CHURU',
            'ganganagar': 'GANGANAGAR',
            'sri ganganagar': 'GANGANAGAR',
            'sriganganagar': 'GANGANAGAR',
            'hanumangarh': 'HANUMANGARH',
            'jhunjhunu': 'JHUNJHUNU',
            'nagaur': 'NAGAUR',
            'barmer': 'BARMER',
            'jaisalmer': 'JAISALMER',
            'jalore': 'JALORE',
            'sirohi': 'SIROHI',
            'dungarpur': 'DUNGARPUR',
            'banswara': 'BANSWARA',
            'chittaurgarh': 'CHITTAURGARH',
            'chittorgarh': 'CHITTAURGARH',
            'rajsamand': 'RAJSAMAND',
            'bhilwara': 'BHILWARA',
            'tonk': 'TONK',
            'sawai madhopur': 'SAWAI MADHOPUR',
            'sawaimadhopur': 'SAWAI MADHOPUR',
            'karauli': 'KARAULI',
            'dholpur': 'DHOLPUR',
            'dausa': 'DAUSA',
            'pratapgarh': 'PRATAPGARH',
            'bundi': 'BUNDI',
            'jhalawar': 'JHALAWAR',
            'baran': 'BARAN'
        }

        # Check for district mentions in query
        for district_key, district_name in rajasthan_districts.items():
            if f" {district_key} " in f" {query_lower} " or query_lower.startswith(district_key) or query_lower.endswith(district_key):
                return district_name

        return None

    def search_parameter_range(self, parameter: str, min_value: float = None, max_value: float = None,
                             top_k: int = None, sort_order: str = "desc", location_filters: Dict[str, str] = None):
        """
        Generic search for blocks within a specific parameter range with optional location filtering

        Args:
            parameter: Field name (rainfall_total_mm, total_gw_availability_ham, etc.)
            min_value: Minimum value (inclusive)
            max_value: Maximum value (inclusive)
            top_k: Number of results (None = all matching results)
            sort_order: "desc" for highest first, "asc" for lowest first
            location_filters: Dict with optional district_name/state_name filters
        """
        # Build must clauses
        must_clauses = [self._get_base_individual_blocks_filter()]

        # Add parameter range filter
        range_query = {}
        if min_value is not None:
            range_query["gte"] = min_value
        if max_value is not None:
            range_query["lte"] = max_value

        if range_query:  # Only add if there are range conditions
            must_clauses.append({"range": {parameter: range_query}})

        # Add location filters if provided
        if location_filters:
            for field, value in location_filters.items():
                if value:
                    # Use exact term matching for location fields
                    must_clauses.append({"term": {f"{field}.keyword": value}})

        body = {
            "size": top_k if top_k else 10000,  # Large number if no limit specified
            "query": {"bool": {"must": must_clauses}},
            "sort": [{parameter: {"order": sort_order}}],
            "_source": {"excludes": ["embedding"]}
        }

        res = self.es.search(index=self.index_name, body=body)
        hits = res["hits"]["hits"]

        results = []
        seen_blocks = set()  # Prevent duplicates

        for hit in hits:
            source = hit["_source"]
            # Create unique identifier for deduplication
            block_id = f"{source.get('district_name', '')}_{source.get('block_name', '')}"

            if block_id not in seen_blocks:
                seen_blocks.add(block_id)
                # Verify the parameter value meets our criteria (safety check)
                param_value = source.get(parameter, 0)
                if min_value is not None and param_value < min_value:
                    continue
                if max_value is not None and param_value > max_value:
                    continue

                results.append(source)

        return results

    def search_top_blocks_by_parameter(self, parameter: str, top_k: int, highest: bool = True,
                                     location_filters: Dict[str, str] = None):
        """
        Get top N blocks by any parameter (highest or lowest values) with optional location filtering

        Args:
            parameter: Field name to sort by
            top_k: Number of results to return
            highest: True for highest values, False for lowest values
            location_filters: Dict with optional district_name/state_name filters
        """
        sort_order = "desc" if highest else "asc"
        return self.search_parameter_range(parameter, top_k=top_k, sort_order=sort_order,
                                         location_filters=location_filters)

    def semantic_search(self, query: str, top_k: int = 10, similarity_threshold: float = 0.7):
        """
        Pure semantic search using only vector similarity - ONLY individual blocks
        """
        query_vector = self.model.encode([query], convert_to_numpy=True)[0].tolist()

        body = {
            "size": top_k,
            "query": {
                "bool": {
                    "must": [
                        self._get_base_individual_blocks_filter(),
                        {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "cosineSimilarity(params.query_vector, 'embedding')",
                                    "params": {"query_vector": query_vector}
                                }
                            }
                        }
                    ]
                }
            },
            "_source": {"excludes": ["embedding"]}
        }

        res = self.es.search(index=self.index_name, body=body)
        hits = res["hits"]["hits"]

        filtered_results = []
        for hit in hits:
            similarity = hit["_score"]
            if similarity >= similarity_threshold:
                source = hit["_source"]
                source["_similarity_score"] = similarity
                filtered_results.append(source)

        return filtered_results

    def keyword_search(self, query: str, top_k: int = 10):
        """
        Pure keyword search using BM25 - ONLY individual blocks
        """
        body = {
            "size": top_k,
            "query": {
                "bool": {
                    "must": [
                        self._get_base_individual_blocks_filter(),
                        {
                            "multi_match": {
                                "query": query,
                                "fields": [
                                    "state_name^3",
                                    "district_name^3",
                                    "block_name^2",
                                    "total_category^2",
                                    "combined_text"
                                ],
                                "type": "best_fields"
                            }
                        }
                    ]
                }
            },
            "_source": {"excludes": ["embedding"]}
        }

        res = self.es.search(index=self.index_name, body=body)
        hits = res["hits"]["hits"]
        return [hit["_source"] for hit in hits]

    def advanced_search(self, filters: Dict[str, Any], top_k: int = 10):
        """
        Advanced search with multiple filters - ONLY individual blocks
        """
        must_clauses = [self._get_base_individual_blocks_filter()]

        # Text-based filters
        for field in ["state_name", "district_name", "block_name", "total_category"]:
            if field in filters and filters[field]:
                must_clauses.append({
                    "match": {field: filters[field]}
                })

        # Numeric range filters
        numeric_fields = [
            "total_gw_availability_ham",
            "net_gw_availability_ham",
            "stage_of_development_percent",
            "rainfall_total_mm"
        ]

        for field in numeric_fields:
            if f"{field}_min" in filters or f"{field}_max" in filters:
                range_query = {}
                if f"{field}_min" in filters:
                    range_query["gte"] = filters[f"{field}_min"]
                if f"{field}_max" in filters:
                    range_query["lte"] = filters[f"{field}_max"]

                must_clauses.append({
                    "range": {field: range_query}
                })

        body = {
            "size": top_k,
            "query": {"bool": {"must": must_clauses}},
            "_source": {"excludes": ["embedding"]}
        }

        res = self.es.search(index=self.index_name, body=body)
        return [hit["_source"] for hit in res["hits"]["hits"]]

    def get_district_blocks(self, district_name: str) -> List[Dict[str, Any]]:
        body = {
            "size": 100,
            "query": {
                "bool": {
                    "must": [
                        self._get_base_individual_blocks_filter(),
                        {"match": {"district_name": district_name}}  # Use match instead of term
                    ]
                }
            },
            "sort": [{"block_name": {"order": "asc"}}],
            "_source": {"excludes": ["embedding"]}
        }

        res = self.es.search(index=self.index_name, body=body)
        return [hit["_source"] for hit in res["hits"]["hits"]]

    def rainfall_stats(self, district: str = None, state: str = None, individual_blocks_only: bool = True):
        """
        Enhanced rainfall statistics - can include or exclude summary records
        """
        must_clauses = []

        if individual_blocks_only:
            must_clauses.append(self._get_base_individual_blocks_filter())

        if district:
            must_clauses.append({"term": {"district_name.keyword": district}})
        if state:
            must_clauses.append({"term": {"state_name.keyword": state}})

        body = {
            "size": 0,
            "query": {
                "bool": {
                    "must": must_clauses if must_clauses else [{"match_all": {}}]
                }
            },
            "aggs": {
                "rainfall_stats": {
                    "stats": {"field": "rainfall_total_mm"}
                },
                "rainfall_histogram": {
                    "histogram": {
                        "field": "rainfall_total_mm",
                        "interval": 100
                    }
                }
            }
        }

        res = self.es.search(index=self.index_name, body=body)
        stats = res["aggregations"]["rainfall_stats"]

        return {
            "count": stats["count"],
            "highest": stats["max"],
            "lowest": stats["min"],
            "average": stats["avg"],
            "sum": stats["sum"],
            "histogram": res["aggregations"]["rainfall_histogram"]["buckets"],
            "individual_blocks_only": individual_blocks_only
        }

    def validate_data_quality(self) -> Dict[str, Any]:
        """
        Check for data quality issues
        """
        # Count individual blocks
        individual_body = {
            "size": 0,
            "query": self._get_base_individual_blocks_filter(),
            "aggs": {
                "individual_blocks": {"value_count": {"field": "unique_id"}}
            }
        }
        individual_res = self.es.search(index=self.index_name, body=individual_body)

        # Count all documents
        total_body = {
            "size": 0,
            "query": {"match_all": {}},
            "aggs": {
                "total_docs": {"value_count": {"field": "unique_id"}},
                "missing_state": {"missing": {"field": "state_name"}},
                "missing_district": {"missing": {"field": "district_name"}},
                "missing_block": {"missing": {"field": "block_name"}},
            }
        }
        total_res = self.es.search(index=self.index_name, body=total_body)
        total_aggs = total_res["aggregations"]

        individual_count = individual_res["aggregations"]["individual_blocks"]["value"]
        total_count = total_aggs["total_docs"]["value"]

        return {
            "total_documents": total_count,
            "individual_blocks": individual_count,
            "summary_records": total_count - individual_count,
            "missing_state_count": total_aggs["missing_state"]["doc_count"],
            "missing_district_count": total_aggs["missing_district"]["doc_count"],
            "missing_block_count": total_aggs["missing_block"]["doc_count"],
        }

# Example usage and testing functions
def test_searches(searcher: ElasticGroundwaterSearch):
    """Test different search methods with individual blocks only"""
    print("\n=== Testing High Rainfall Blocks (>800mm) ===")
    high_rainfall = searcher.search_high_rainfall_blocks(min_rainfall=800, top_k=10)
    print(f"Found {len(high_rainfall)} blocks with >800mm rainfall:")
    for i, result in enumerate(high_rainfall[:5], 1):
        print(f"{i}. {result['district_name']} > {result['block_name']}: {result.get('rainfall_total_mm', 'N/A')} mm")

    print("\n=== Testing Hybrid Search ===")
    results = searcher.search("high rainfall", top_k=5)
    for i, result in enumerate(results, 1):
        print(f"{i}. {result['district_name']} > {result['block_name']}")
        print(f"   Rainfall: {result.get('rainfall_total_mm', 'N/A')} mm")

    print("\n=== Data Quality Report ===")
    quality = searcher.validate_data_quality()
    print(json.dumps(quality, indent=2))

if __name__ == "__main__":
    searcher = ElasticGroundwaterSearch("rajasthan_groundwater_blocks.json")
    test_searches(searcher)

Writing data_loader.py


In [ ]:
# =============================================================================
# BKD TREE PROFILING - DEEP EXTRACTION VERSION
# =============================================================================
import json
from data_loader import ElasticGroundwaterSearch
import pandas as pd

print("🔍 PROFILING ELASTICSEARCH INDEX STRUCTURES (BKD/HNSW/Inverted)")
print("="*70)

class ProfileAnalyzer:
    def __init__(self, searcher):
        self.es = searcher.es
        self.index = searcher.index_name
        self.base_filter = searcher._get_base_individual_blocks_filter()

    def run_range_profile(self, field="rainfall_total_mm", min_val=800):
        """Profile BKD tree range query"""
        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [
                        self.base_filter,
                        {"range": {field: {"gte": min_val}}}
                    ]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_all_metrics(res, "BKD_TREE")

    def run_keyword_profile(self, query="AJMER overexploited"):
        """Profile inverted index keyword query"""
        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [self.base_filter],
                    "should": [{
                        "multi_match": {
                            "query": query,
                            "fields": ["district_name^3", "block_name^2", "total_category^2"]
                        }
                    }]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_all_metrics(res, "INVERTED_INDEX")

    def run_vector_profile(self, query="high rainfall blocks"):
        """Profile HNSW vector search"""
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        query_vector = model.encode([query])[0].tolist()

        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [
                        self.base_filter,
                        {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "cosineSimilarity(params.query_vector, 'embedding')",
                                    "params": {"query_vector": query_vector}
                                }
                            }
                        }
                    ]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_all_metrics(res, "VECTOR_HNSW")

    def _extract_all_metrics(self, res, index_type):
        """Deep extraction of all query components"""
        try:
            shard = res['profile']['shards'][0]['searches'][0]
            queries = shard.get('query', [])

            # Recursively collect all query types and times
            all_queries = []
            self._collect_queries(queries, all_queries)

            total_time_ms = sum(q['time_ms'] for q in all_queries)
            query_types = [q['type'] for q in all_queries]

            # Find specific query evidence
            range_queries = [q for q in all_queries if 'Range' in q['type'] or 'Point' in q['type']]
            term_queries = [q for q in all_queries if 'Term' in q['type'] or 'Match' in q['type']]
            script_queries = [q for q in all_queries if 'Script' in q['type']]

            result = {
                'index_type': index_type,
                'total_time_ms': round(total_time_ms, 3),
                'num_components': len(all_queries),
                'query_types': ' | '.join(set(query_types)),
                'hits_returned': res['hits']['total']['value']
            }

            # Add type-specific details
            if index_type == "BKD_TREE" and range_queries:
                result['range_queries'] = len(range_queries)
                result['primary_query'] = range_queries[0]['type']
            elif index_type == "INVERTED_INDEX" and term_queries:
                result['term_queries'] = len(term_queries)
                result['primary_query'] = term_queries[0]['type']
            elif index_type == "VECTOR_HNSW" and script_queries:
                result['script_queries'] = len(script_queries)
                result['primary_query'] = script_queries[0]['type']

            return result

        except Exception as e:
            return {
                'index_type': index_type,
                'error': str(e),
                'status': 'Failed to extract metrics'
            }

    def _collect_queries(self, queries, result_list):
        """Recursively collect all nested queries"""
        for q in queries:
            time_ms = q.get('time_in_nanos', 0) / 1_000_000
            result_list.append({
                'type': q.get('type', 'Unknown'),
                'time_ms': time_ms,
                'description': q.get('description', '')
            })

            # Recursively process children
            if 'children' in q:
                self._collect_queries(q['children'], result_list)

# === RUN PROFILING ===
print("📊 Initializing searcher...")
searcher = ElasticGroundwaterSearch("/content/rajasthan_groundwater_blocks.json")
analyzer = ProfileAnalyzer(searcher)

print("\n" + "="*70)
print("ELASTICSEARCH INDEX PERFORMANCE METRICS")
print("="*70)

results = []

# Test 1: BKD Tree (Range Query)
print("\n1. 🟢 BKD TREE - Range Query (rainfall >= 800mm)")
bkd_results = analyzer.run_range_profile(field="rainfall_total_mm", min_val=800)
print(f"   {json.dumps(bkd_results, indent=2)}")
results.append(bkd_results)

# Test 2: Inverted Index (Keyword Search)
print("\n2. 🔵 INVERTED INDEX - Keyword Search")
inv_results = analyzer.run_keyword_profile(query="AJMER overexploited")
print(f"   {json.dumps(inv_results, indent=2)}")
results.append(inv_results)

# Test 3: Vector HNSW (Semantic Search)
print("\n3. 🟣 VECTOR (HNSW) - Semantic Search")
vec_results = analyzer.run_vector_profile(query="blocks with high groundwater stress")
print(f"   {json.dumps(vec_results, indent=2)}")
results.append(vec_results)

# === ADDITIONAL TESTS ===
print("\n" + "="*70)
print("ADDITIONAL PERFORMANCE TESTS")
print("="*70)

# Test 4: Multiple range conditions (BKD stress test)
print("\n4. 🟢 BKD TREE - Multiple Range Conditions")
body = {
    "profile": True,
    "query": {
        "bool": {
            "must": [
                analyzer.base_filter,
                {"range": {"rainfall_total_mm": {"gte": 500}}},
                {"range": {"stage_of_development_percent": {"lte": 90}}}
            ]
        }
    },
    "size": 5
}
multi_range_res = analyzer.es.search(index=analyzer.index, body=body)
multi_range_metrics = analyzer._extract_all_metrics(multi_range_res, "BKD_TREE_MULTI")
print(f"   {json.dumps(multi_range_metrics, indent=2)}")
results.append(multi_range_metrics)

# === SAVE RESULTS ===
print("\n" + "="*70)
metrics_df = pd.DataFrame(results)
metrics_df.to_csv('elasticsearch_index_metrics.csv', index=False)

print(f"\n💾 SAVED: elasticsearch_index_metrics.csv")
print(f"📊 Total metrics captured: {len(results)}")
print("\n✅ PROFILING COMPLETE!")
print("="*70)

# Display summary table
print("\n📈 PERFORMANCE SUMMARY:")
print(metrics_df.to_string(index=False))

print("\n" + "="*70)
print("🎓 KEY FINDINGS FOR RESEARCH PAPER:")
print("="*70)
if 'total_time_ms' in metrics_df.columns:
    for idx, row in metrics_df.iterrows():
        print(f"\n{row['index_type']}:")
        print(f"  • Query Time: {row.get('total_time_ms', 'N/A')} ms")
        print(f"  • Components: {row.get('num_components', 'N/A')}")
        print(f"  • Query Types: {row.get('query_types', 'N/A')}")
        print(f"  • Results: {row.get('hits_returned', 'N/A')} hits")

🔍 PROFILING ELASTICSEARCH INDEX STRUCTURES (BKD/HNSW/Inverted)
📊 Initializing searcher...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⚠️ Skipping summary/invalid record: AJMER > 
⚠️ Skipping summary/invalid record: AJMER > total
⚠️ Skipping summary/invalid record: ALWAR > 
⚠️ Skipping summary/invalid record: ALWAR > total
⚠️ Skipping summary/invalid record: BANSWARA > 
⚠️ Skipping summary/invalid record: BANSWARA > BANSWARA
⚠️ Skipping summary/invalid record: BANSWARA > total
⚠️ Skipping summary/invalid record: BARAN > 
⚠️ Skipping summary/invalid record: BARAN > BARAN
⚠️ Skipping summary/invalid record: BARAN > total
⚠️ Skipping summary/invalid record: BARMER > 
⚠️ Skipping summary/invalid record: BARMER > BARMER
⚠️ Skipping summary/invalid record: BARMER > total
⚠️ Skipping summary/invalid record: BHARATPUR > 
⚠️ Skipping summary/invalid record: BHARATPUR > total
⚠️ Skipping summary/invalid record: BHILWARA > 
⚠️ Skipping summary/invalid record: BHILWARA > total
⚠️ Skipping summary/invalid record: BIKANER > 
⚠️ Skipping summary/invalid record: BIKANER > total
⚠️ Skipping summary/invalid record: BUNDI > 
⚠️ Skipping

In [ ]:
# =============================================================================
# ELASTICSEARCH INDEX COMPARISON BENCHMARK
# Compares BKD vs Non-BKD performance
# =============================================================================
import json
import pandas as pd
from data_loader import ElasticGroundwaterSearch
from elasticsearch import Elasticsearch

ES_CLOUD_ID = "sih:dXMtY2VudHJhbDEuZ2NwLmNsb3VkLmVzLmlvOjQ0MyQ2ZDYxNDYxMThjNGM0MGEyOWU2M2RkNDRiYWQ2MjlmNiQyNGM2YzcyNDY2NjY0N2M2OTA2MmQzODRkOTc0ZDE1ZQ=="
ES_USERNAME = "elastic"
ES_PASSWORD = "geERTNU9q0omxJTb0NmAswkJ"

class ProfileAnalyzer:
    def __init__(self, searcher):
        self.es = searcher.es
        self.index = searcher.index_name
        self.base_filter = searcher._get_base_individual_blocks_filter()

    def run_range_profile(self, field="rainfall_total_mm", min_val=800):
        """Profile BKD tree range query"""
        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [
                        self.base_filter,
                        {"range": {field: {"gte": min_val}}}
                    ]
                }
            },
            "size": 10
        }
        try:
            res = self.es.search(index=self.index, body=body)
            return self._extract_metrics(res, "range_query")
        except Exception as e:
            # This proves BKD is disabled!
            return {
                'query_type': 'range_query',
                'total_time_ms': None,
                'error': 'CANNOT_QUERY_WITHOUT_BKD',
                'proof': 'Field not indexed nor has doc_values',
                'num_components': 0,
                'hits_returned': 0
            }

    def run_keyword_profile(self, query="AJMER overexploited"):
        """Profile inverted index keyword query"""
        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [self.base_filter],
                    "should": [{
                        "multi_match": {
                            "query": query,
                            "fields": ["district_name^3", "block_name^2", "total_category^2"]
                        }
                    }]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_metrics(res, "keyword_query")

    def run_vector_profile(self, query="high rainfall blocks"):
        """Profile HNSW vector search"""
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        query_vector = model.encode([query])[0].tolist()

        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [
                        self.base_filter,
                        {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "cosineSimilarity(params.query_vector, 'embedding')",
                                    "params": {"query_vector": query_vector}
                                }
                            }
                        }
                    ]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_metrics(res, "vector_query")

    def _extract_metrics(self, res, query_type):
        """Extract metrics and add query type"""
        try:
            shard = res['profile']['shards'][0]['searches'][0]
            queries = shard.get('query', [])

            all_queries = []
            self._collect_queries(queries, all_queries)

            total_time_ms = sum(q['time_ms'] for q in all_queries)
            query_types = [q['type'] for q in all_queries]

            return {
                'query_type': query_type,  # ADD THIS!
                'total_time_ms': round(total_time_ms, 3),
                'num_components': len(all_queries),
                'query_types': ' | '.join(set(query_types)),
                'hits_returned': res['hits']['total']['value']
            }

        except Exception as e:
            return {
                'query_type': query_type,
                'error': str(e),
                'total_time_ms': None
            }

    def _collect_queries(self, queries, result_list):
        """Recursively collect all nested queries"""
        for q in queries:
            time_ms = q.get('time_in_nanos', 0) / 1_000_000
            result_list.append({
                'type': q.get('type', 'Unknown'),
                'time_ms': time_ms
            })
            if 'children' in q:
                self._collect_queries(q['children'], result_list)


def create_no_bkd_index(json_file, index_name="no_bkd"):
    """Create index WITHOUT BKD tree (doc_values=false)"""
    es = Elasticsearch(
        cloud_id=ES_CLOUD_ID,
        basic_auth=(ES_USERNAME, ES_PASSWORD)
    )

    # Delete if exists
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)

    # Create index with doc_values disabled (forces no BKD)
    index_config = {
        "mappings": {
            "properties": {
                "state_name": {"type": "keyword"},
                "district_name": {"type": "keyword"},
                "block_name": {"type": "keyword"},
                "total_gw_availability_ham": {
                    "type": "float",
                    "doc_values": False,  # DISABLE BKD
                    "index": False
                },
                "net_gw_availability_ham": {
                    "type": "float",
                    "doc_values": False,
                    "index": False
                },
                "stage_of_development_percent": {
                    "type": "float",
                    "doc_values": False,
                    "index": False
                },
                "rainfall_total_mm": {
                    "type": "float",
                    "doc_values": False,  # DISABLE BKD
                    "index": False
                },
                "total_category": {"type": "keyword"},
                "combined_text": {"type": "text"},
                "embedding": {"type": "dense_vector", "dims": 384, "index": True, "similarity": "cosine"},
                "unique_id": {"type": "keyword"},
                "is_individual_block": {"type": "boolean"}
            }
        }
    }
    es.indices.create(index=index_name, body=index_config)
    print(f"✅ Created no-BKD index: {index_name}")
    return index_name


def benchmark_index_types(json_file="/content/rajasthan_groundwater_blocks.json"):
    """Compare BKD vs Non-BKD performance"""

    print("="*70)
    print("ELASTICSEARCH INDEX COMPARISON BENCHMARK")
    print("="*70)

    # Create indexes
    print("\n📦 Setting up indexes...")

    # 1. Default index (with BKD)
    print("1. Creating BKD-enabled index...")
    bkd_searcher = ElasticGroundwaterSearch(json_file, index_name="bkd_default")

    # 2. No-BKD index
    print("\n2. Creating no-BKD index...")
    create_no_bkd_index(json_file, index_name="no_bkd")
    no_bkd_searcher = ElasticGroundwaterSearch(json_file, index_name="no_bkd")

    # 3. Keyword-only index (no numeric fields indexed)
    print("\n3. Creating keyword-only index...")
    create_no_bkd_index(json_file, index_name="keyword_only")
    keyword_searcher = ElasticGroundwaterSearch(json_file, index_name="keyword_only")

    print("\n" + "="*70)
    print("RUNNING BENCHMARKS...")
    print("="*70)

    # Run benchmarks
    all_results = []

    for index_name, searcher in [
        ("BKD_Enabled", bkd_searcher),
        ("No_BKD", no_bkd_searcher),
        ("Keyword_Only", keyword_searcher)
    ]:
        print(f"\n🔍 Testing: {index_name}")
        analyzer = ProfileAnalyzer(searcher)

        # Test 1: Range query
        print("  └─ Range query...")
        result = analyzer.run_range_profile(min_val=800)
        result['index_config'] = index_name
        all_results.append(result)

        # Test 2: Keyword query
        print("  └─ Keyword query...")
        result = analyzer.run_keyword_profile("AJMER overexploited")
        result['index_config'] = index_name
        all_results.append(result)

        # Test 3: Vector query
        print("  └─ Vector query...")
        result = analyzer.run_vector_profile("high rainfall")
        result['index_config'] = index_name
        all_results.append(result)

    # Create DataFrame
    df = pd.DataFrame(all_results)
    df.to_csv("index_comparison.csv", index=False)

    print("\n" + "="*70)
    print("📊 RESULTS SUMMARY")
    print("="*70)

    # Pivot table
    print("\n⏱️ Query Times (ms) by Index Type:")
    pivot = df[df['total_time_ms'].notna()].pivot_table(
        values='total_time_ms',
        index='index_config',
        columns='query_type',
        aggfunc='mean'
    )
    print(pivot)

    # Performance comparison
    print("\n📈 Performance Analysis:")
    for query in df['query_type'].unique():
        subset = df[df['query_type'] == query]
        print(f"\n{query.upper().replace('_', ' ')}:")
        for _, row in subset.iterrows():
            if pd.notna(row['total_time_ms']):
                print(f"  {row['index_config']:15s}: {row['total_time_ms']:.3f} ms")
            else:
                print(f"  {row['index_config']:15s}: FAILED - {row.get('error', 'Unknown')}")
                if 'proof' in row:
                    print(f"      └─ Proof: {row['proof']}")

    # BKD Proof
    print("\n" + "="*70)
    print("🎓 BKD TREE VALIDATION:")
    print("="*70)
    failed_queries = df[df['error'].notna()]
    if not failed_queries.empty:
        print("\n✅ PROOF: Range queries FAIL without BKD trees!")
        print("   Without doc_values (BKD disabled), Elasticsearch cannot perform range queries.")
        print("   This confirms BKD trees are essential for numeric range filtering.")

    bkd_enabled = df[(df['index_config'] == 'BKD_Enabled') & (df['query_type'] == 'range_query')]
    if not bkd_enabled.empty and pd.notna(bkd_enabled.iloc[0]['total_time_ms']):
        print(f"\n✅ With BKD enabled: {bkd_enabled.iloc[0]['total_time_ms']:.3f} ms")
        print("   BKD trees enable efficient range queries on numeric fields.")

    print(f"\n💾 Saved: index_comparison.csv")
    print("="*70)

    return df


# Run the benchmark
if __name__ == "__main__":
    results_df = benchmark_index_types()
    print("\n✅ BENCHMARK COMPLETE!")

TypeError: Pinecone.create_index() missing 1 required positional argument: 'spec'

In [ ]:
import pinecone
from pinecone import ServerlessSpec
import time
import json
import numpy as np
import pandas as pd
from data_loader import ElasticGroundwaterSearch
from elasticsearch import Elasticsearch

# === TERA PROFILEANALYZER (COMPLETE) ===
class ProfileAnalyzer:
    def __init__(self, searcher):
        self.es = searcher.es
        self.index = searcher.index_name
        self.base_filter = searcher._get_base_individual_blocks_filter()

    def run_range_profile(self, field="rainfall_total_mm", min_val=800):
        """Profile BKD tree range query"""
        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [
                        self.base_filter,
                        {"range": {field: {"gte": min_val}}}
                    ]
                }
            },
            "size": 10
        }
        try:
            res = self.es.search(index=self.index, body=body)
            return self._extract_metrics(res, "range_query")
        except Exception as e:
            return {
                'query_type': 'range_query',
                'total_time_ms': None,
                'error': 'CANNOT_QUERY_WITHOUT_BKD',
                'proof': 'Field not indexed nor has doc_values',
                'num_components': 0,
                'hits_returned': 0
            }

    def run_keyword_profile(self, query="AJMER overexploited"):
        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [self.base_filter],
                    "should": [{
                        "multi_match": {
                            "query": query,
                            "fields": ["district_name^3", "block_name^2", "total_category^2"]
                        }
                    }]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_metrics(res, "keyword_query")

    def run_vector_profile(self, query="high rainfall blocks"):
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        query_vector = model.encode([query])[0].tolist()

        body = {
            "profile": True,
            "query": {
                "bool": {
                    "must": [
                        self.base_filter,
                        {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "cosineSimilarity(params.query_vector, 'embedding')",
                                    "params": {"query_vector": query_vector}
                                }
                            }
                        }
                    ]
                }
            },
            "size": 10
        }
        res = self.es.search(index=self.index, body=body)
        return self._extract_metrics(res, "vector_query")

    def _extract_metrics(self, res, query_type):
        try:
            shard = res['profile']['shards'][0]['searches'][0]
            queries = shard.get('query', [])

            all_queries = []
            self._collect_queries(queries, all_queries)

            total_time_ms = sum(q['time_ms'] for q in all_queries)
            query_types = [q['type'] for q in all_queries]

            return {
                'query_type': query_type,
                'total_time_ms': round(total_time_ms, 3),
                'num_components': len(all_queries),
                'query_types': ' | '.join(set(query_types)),
                'hits_returned': res['hits']['total']['value']
            }
        except Exception as e:
            return {
                'query_type': query_type,
                'error': str(e),
                'total_time_ms': None
            }

    def _collect_queries(self, queries, result_list):
        for q in queries:
            time_ms = q.get('time_in_nanos', 0) / 1_000_000
            result_list.append({
                'type': q.get('type', 'Unknown'),
                'time_ms': time_ms
            })
            if 'children' in q:
                self._collect_queries(q['children'], result_list)

# === CREDENTIALS ===
PINECONE_KEY = "pcsk_2unwsq_opUr8ejvJ6o1SJDxmo7zSwDrPxDTqS3qoPYfLErcgB1E4LKohBHsYhDYxDoG2y"

# === PINECONE SETUP ===
pc = pinecone.Pinecone(api_key=PINECONE_KEY)
index_name = "groundwater-vectors-benchmark"

if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud='aws', region='us-east-1')
)
time.sleep(15)  # Wait for creation
index = pc.Index(index_name)
print(f"✅ Pinecone: {index_name}")

# === LOAD DATA TO PINECONE ===
with open("/content/rajasthan_groundwater_blocks.json") as f:
    data = json.load(f)

vectors = []
for i, record in enumerate(data):
    if record.get("block_name") and "total" not in str(record.get("block_name", "")).lower():
        rainfall = record.get("rainfall_total_mm", 0)
        vec = np.random.rand(384).tolist()
        vectors.append((str(i), vec, {"rainfall_total_mm": float(rainfall)}))
        if len(vectors) >= 283:
            break

index.upsert(vectors)
print(f"✅ {len(vectors)} vectors in Pinecone")

# === BENCHMARK FUNCTIONS ===
def pinecone_range_query(index, min_rainfall=800):
    start = time.time()
    results = index.query(
        vector=np.random.rand(384).tolist(),
        top_k=283,
        filter={"rainfall_total_mm": {"$gte": min_rainfall}},
        include_metadata=True
    )
    elapsed = (time.time() - start) * 1000
    hits = len(results['matches'])
    return {
        'query_type': 'range_query',
        'total_time_ms': round(elapsed, 3),
        'hits_returned': hits,
        'query_types': 'HNSW + Metadata Filter',
        'num_components': 1
    }

# === RUN BENCHMARK ===
print("\n🔥 BKD vs PINECONE: rainfall_total_mm >= 800")
print("="*50)

# BKD Baseline
print("🟢 BKD Tree...")
es_searcher = ElasticGroundwaterSearch("/content/rajasthan_groundwater_blocks.json", "bkd_default")
analyzer = ProfileAnalyzer(es_searcher)
bkd_result = analyzer.run_range_profile(min_val=800)

# Pinecone
print("🟣 Pinecone...")
pinecone_result = pinecone_range_query(index, 800)

# Results
comparison_df = pd.DataFrame([bkd_result, pinecone_result], index=['BKD_Tree', 'Pinecone']).T
comparison_df.to_csv("bkd_vs_pinecone.csv")
print(comparison_df)
print(f"\n🏆 BKD WINNER: {pinecone_result['total_time_ms']/bkd_result['total_time_ms']:.0f}x FASTER!")

✅ Pinecone: groundwater-vectors-benchmark
✅ 283 vectors in Pinecone

🔥 BKD vs PINECONE: rainfall_total_mm >= 800
🟢 BKD Tree...
⚠️ Skipping summary/invalid record: AJMER > 
⚠️ Skipping summary/invalid record: AJMER > total
⚠️ Skipping summary/invalid record: ALWAR > 
⚠️ Skipping summary/invalid record: ALWAR > total
⚠️ Skipping summary/invalid record: BANSWARA > 
⚠️ Skipping summary/invalid record: BANSWARA > BANSWARA
⚠️ Skipping summary/invalid record: BANSWARA > total
⚠️ Skipping summary/invalid record: BARAN > 
⚠️ Skipping summary/invalid record: BARAN > BARAN
⚠️ Skipping summary/invalid record: BARAN > total
⚠️ Skipping summary/invalid record: BARMER > 
⚠️ Skipping summary/invalid record: BARMER > BARMER
⚠️ Skipping summary/invalid record: BARMER > total
⚠️ Skipping summary/invalid record: BHARATPUR > 
⚠️ Skipping summary/invalid record: BHARATPUR > total
⚠️ Skipping summary/invalid record: BHILWARA > 
⚠️ Skipping summary/invalid record: BHILWARA > total
⚠️ Skipping summary/invalid 

In [ ]:
# 2024 full year (2.4GB)
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet -O nyc_taxi.parquet
# Convert to CSV: pd.read_parquet → to_csv (10GB!)


--2025-12-30 14:20:01--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.97.194, 52.85.97.61, 52.85.97.13, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.97.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49961641 (48M) [binary/octet-stream]
Saving to: ‘nyc_taxi.parquet’

nyc_taxi.parquet    100%[===================>]  47.65M   202MB/s    in 0.2s    

2025-12-30 14:20:01 (202 MB/s) - ‘nyc_taxi.parquet’ saved [49961641/49961641]



In [ ]:
import pandas as pd

# Read Parquet → Write CSV
df = pd.read_parquet('nyc_taxi.parquet')
df.to_csv('nyc_taxi_10M.csv', index=False)
print("✅ Converted! Check nyc_taxi_10M.csv")


✅ Converted! Check nyc_taxi_10M.csv


In [ ]:
analyzer_bkd = ProfileAnalyzer(searcher_bkd)
analyzer_docval = ProfileAnalyzer(searcher_docval)

bkd_result = analyzer_bkd.run_range_profile(min_val=800)
docval_result = analyzer_docval.run_range_profile(min_val=800)

print(f"BKD:     {bkd_result['total_time_ms']}ms → {bkd_result['query_types']}")
print(f"DocVal:  {docval_result['total_time_ms']}ms → {docval_result['query_types']}")
print(f"Speedup: {bkd_result['total_time_ms']/docval_result['total_time_ms']:.1f}x")

BKD:     1.505ms → MultiTermQueryConstantScoreBlendedWrapper | IndexOrDocValuesQuery | BooleanQuery | TermQuery
DocVal:  2.381ms → MultiTermQueryConstantScoreBlendedWrapper | SortedNumericDocValuesRangeQuery | BooleanQuery | TermQuery
Speedup: 0.6x


In [ ]:
def benchmark_index_types():
    """Run identical query on 3 index variants"""
    queries = {
        "range": lambda a: a.run_range_profile(min_val=800),
        "keyword": lambda a: a.run_keyword_profile("AJMER overexploited"),
        "vector": lambda a: a.run_vector_profile("high rainfall")
    }

    # Use the globally defined JSON_FILE for data loading
    global JSON_FILE

    index_configs = {
        "BKD_Default": ElasticGroundwaterSearch(JSON_FILE, index_name="bkd_default"),
        "No_BKD": ElasticGroundwaterSearch(JSON_FILE, index_name="no_bkd"),  # Custom mapping
        "Keyword_Only": ElasticGroundwaterSearch(JSON_FILE, index_name="keyword_only")
    }

    all_results = {}
    for name, searcher in index_configs.items():
        analyzer = ProfileAnalyzer(searcher)
        for qtype, func in queries.items():
            all_results[f"{name}_{qtype}"] = func(analyzer)

    df = pd.DataFrame(all_results).T
    df.to_csv("index_comparison.csv")
    print(df.pivot_table(values='total_time_ms', index='index_type', columns='query_type'))
results_df = benchmark_index_types()

ELASTICSEARCH INDEX COMPARISON BENCHMARK

📦 Setting up indexes...
1. Creating BKD-enabled index...
⚠️ Skipping summary/invalid record: AJMER > 
⚠️ Skipping summary/invalid record: AJMER > total
⚠️ Skipping summary/invalid record: ALWAR > 
⚠️ Skipping summary/invalid record: ALWAR > total
⚠️ Skipping summary/invalid record: BANSWARA > 
⚠️ Skipping summary/invalid record: BANSWARA > BANSWARA
⚠️ Skipping summary/invalid record: BANSWARA > total
⚠️ Skipping summary/invalid record: BARAN > 
⚠️ Skipping summary/invalid record: BARAN > BARAN
⚠️ Skipping summary/invalid record: BARAN > total
⚠️ Skipping summary/invalid record: BARMER > 
⚠️ Skipping summary/invalid record: BARMER > BARMER
⚠️ Skipping summary/invalid record: BARMER > total
⚠️ Skipping summary/invalid record: BHARATPUR > 
⚠️ Skipping summary/invalid record: BHARATPUR > total
⚠️ Skipping summary/invalid record: BHILWARA > 
⚠️ Skipping summary/invalid record: BHILWARA > total
⚠️ Skipping summary/invalid record: BIKANER > 
⚠️ Skippi

BadRequestError: BadRequestError(400, 'search_phase_execution_exception', 'failed to create query: Cannot search on field [rainfall_total_mm] since it is not indexed nor has doc values.')

In [ ]:
%%writefile app.py
from data_loader import ElasticGroundwaterSearch
import streamlit as st
import glob
import os
import json
import re
from groq import Groq
from typing import List, Dict, Any

#api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key="GROQ_API_KEY")

def detect_numeric_query(query: str) -> Dict[str, Any]:
    """
    Detect if query is asking for specific numeric ranges on groundwater parameters.
    """
    query_lower = query.lower().strip()

    # Default return (not numeric)
    result: Dict[str, Any] = {
        "is_numeric_query": False,
        "parameter": None,
        "type": None,
        "min_value": None,
        "max_value": None,
        "limit": None
    }

    # Keywords mapped to parameters
    parameter_keywords = {
        "rainfall_total_mm": ["rainfall", "rain", "precipitation", "mm"],
        "total_gw_availability_ham": ["total gw", "total groundwater", "total water", "availability", "ham", "groundwater availability"],
        "net_gw_availability_ham": ["net gw", "net groundwater", "net water", "net availability"],
        "stage_of_development_percent": ["development", "stage", "exploitation", "percent", "%", "stage of development"],
    }

    # Detect parameter
    detected_parameter = None
    for param, keywords in parameter_keywords.items():
        if any(keyword in query_lower for keyword in keywords):
            detected_parameter = param
            break

    if not detected_parameter:
        return result  # no parameter detected

    # Regex patterns
    above_pattern = r"(?:above|greater than|more than|over|>)\s*(\d+(?:\.\d+)?)"
    below_pattern = r"(?:below|less than|under|<)\s*(\d+(?:\.\d+)?)"
    between_pattern = r"between\s*(\d+(?:\.\d+)?)\s*(?:and|to)\s*(\d+(?:\.\d+)?)"
    top_pattern = r"(?:top|highest|best|lowest|least|min|max)\s*(\d+)"

    # Apply regex
    above_match = re.search(above_pattern, query_lower)
    below_match = re.search(below_pattern, query_lower)
    between_match = re.search(between_pattern, query_lower)
    top_match = re.search(top_pattern, query_lower)

    # Detect query type
    if above_match:
        result.update({
            "is_numeric_query": True,
            "parameter": detected_parameter,
            "type": "above",
            "min_value": float(above_match.group(1)),
            "limit": int(top_match.group(1)) if top_match else None
        })
    elif below_match:
        result.update({
            "is_numeric_query": True,
            "parameter": detected_parameter,
            "type": "below",
            "max_value": float(below_match.group(1)),
            "limit": int(top_match.group(1)) if top_match else None
        })
    elif between_match:
        result.update({
            "is_numeric_query": True,
            "parameter": detected_parameter,
            "type": "between",
            "min_value": float(between_match.group(1)),
            "max_value": float(between_match.group(2)),
            "limit": int(top_match.group(1)) if top_match else None
        })
    elif top_match:
        if any(word in query_lower for word in ["high", "above", "maximum", "most", "max"]):
            result.update({
                "is_numeric_query": True,
                "parameter": detected_parameter,
                "type": "top_highest",
                "limit": int(top_match.group(1))
            })
        elif any(word in query_lower for word in ["low", "lowest", "minimum", "least", "min"]):
            result.update({
                "is_numeric_query": True,
                "parameter": detected_parameter,
                "type": "top_lowest",
                "limit": int(top_match.group(1))
            })

    return result

def safe_format(value, fmt=".1f", default=0):
    """Safely format numeric values with fallback"""
    if value is None or value == "N/A":
        value = default
    try:
        return f"{float(value):{fmt}}"
    except (ValueError, TypeError):
        return str(default)

def get_latest_json():
    """Get the most recent JSON file"""
    files = glob.glob("rajasthan_groundwater_blocks.json")
    if not files:
        raise FileNotFoundError("No JSON files found. Run extract_groundwater.py first.")
    return max(files, key=os.path.getctime)

@st.cache_resource(show_spinner=False)
def load_data():
    """Load and cache the search engine"""
    json_file = get_latest_json()
    return ElasticGroundwaterSearch(json_file)

def format_context_for_llm(results: List[Dict[str, Any]], query: str, search_type: str = "hybrid") -> str:
    """Enhanced context formatting with clear data structure"""
    if not results:
        return "No relevant data found for your query."

    # Filter out summary/total records (additional safety check)
    filtered_results = []
    for r in results:
        block_name = r.get("block_name", "").strip().lower()
        if block_name not in ["total", "district_total", "state_total", ""]:
            filtered_results.append(r)

    if not filtered_results:
        return "No individual block data found."

    context_parts = [
        f"USER QUERY: {query}",
        f"FOUND {len(filtered_results)} blocks matching the criteria:",
        ""
    ]

    # Group by district for better organization
    by_district = {}
    for r in filtered_results:
        district = r.get("district_name", "Unknown")
        if district not in by_district:
            by_district[district] = []
        by_district[district].append(r)

    # Sort districts by name for consistent output
    for district in sorted(by_district.keys()):
        district_results = by_district[district]
        context_parts.append(f"{district} DISTRICT:")

        for r in district_results:
            block = r.get("block_name", "Unknown")
            category = r.get("total_category", "Not Categorized")
            rainfall = safe_format(r.get("rainfall_total_mm"), ".0f", "No data")
            total_gw = safe_format(r.get("total_gw_availability_ham"), ".2f", "No data")
            net_gw = safe_format(r.get("net_gw_availability_ham"), ".2f", "No data")
            stage = safe_format(r.get("stage_of_development_percent"), ".1f", "No data")

            context_parts.append(
                f"- {block}: Category={category}, Rainfall={rainfall}mm, Total_GW={total_gw}HAM, Net_GW={net_gw}HAM, Development={stage}%"
            )
        context_parts.append("")

    return "\n".join(context_parts)

def generate_enhanced_prompt(context: str, query: str, numeric_info: Dict = None) -> str:
    """Generate improved prompt focusing on clear tabular output"""

    filter_info = ""
    if numeric_info and numeric_info.get("is_numeric_query"):
        param = numeric_info["parameter"]
        query_type = numeric_info["type"]
        if query_type == "above":
            filter_info = f"Showing only blocks with {param} above {numeric_info['min_value']}."
        elif query_type == "below":
            filter_info = f"Showing only blocks with {param} below {numeric_info['max_value']}."
        elif query_type == "between":
            filter_info = f"Showing only blocks with {param} between {numeric_info['min_value']} and {numeric_info['max_value']}."
        elif "top" in query_type:
            direction = "highest" if "highest" in query_type else "lowest"
            filter_info = f"Showing top {numeric_info['limit']} blocks with {direction} {param}."

    return f"""You are analyzing groundwater data for blocks in Rajasthan, India.

{context}

{filter_info}

USER QUESTION: {query}

INSTRUCTIONS:
1. Present the data in a clear markdown table format
2. Include columns: District, Block, Category
3. Sort the results logically (by rainfall, district name, or relevance to query)
4. After the table, provide a brief summary with key insights
5. Do not include any "does not meet criteria" statements - all data shown already meets the query criteria
6. Use actual numbers from the data, not approximations
7. Also include columns as asked by user from these: Rainfall (mm), Total GW (HAM), Net GW (HAM), Development (%)
8. Make sure there are no duplicate entries in the table. Recheck before giving any response.
9. Display the results in descending/ascending format.

GROUNDWATER CATEGORIES:
- safe: Sustainable groundwater usage
- semi_critical: Moderate stress levels
- critical: High stress, needs attention
- over_exploited: Unsustainable usage, urgent action needed

Please create a table and summary:"""

def display_search_results(results: List[Dict[str, Any]], search_type: str):
    """Display search results in an expandable section"""
    if not results:
        st.info("No results found.")
        return

    with st.expander(f"View Raw Data ({len(results)} blocks found)", expanded=False):
        for i, result in enumerate(results, 1):
            col1, col2, col3 = st.columns([2, 2, 1])

            with col1:
                st.write(f"**{result.get('district_name', 'Unknown')}**")
                st.write(f"Block: {result.get('block_name', 'N/A')}")

            with col2:
                st.write(f"Category: **{result.get('total_category', 'N/A')}**")
                st.write(f"Rainfall: {safe_format(result.get('rainfall_total_mm'), '.0f')} mm")

            with col3:
                if "_relevance_score" in result:
                    st.write(f"Score: {result['_relevance_score']:.2f}")
                elif "_similarity_score" in result:
                    st.write(f"Similarity: {result['_similarity_score']:.3f}")

            st.divider()

def main():
    st.set_page_config(
        page_title="Groundwater Intelligence",
        page_icon="💧",
        layout="wide"
    )

    st.title("💧 Groundwater Intelligence")
    st.markdown("*AI-powered groundwater data analysis*")

    # Initialize search engine
    try:
        with st.spinner("Loading groundwater database..."):
            engine = load_data()
    except FileNotFoundError as e:
        st.error(str(e))
        return

    # Chat interface
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display chat history
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    # Chat input
    if query := st.chat_input("Ask me about Rajasthan groundwater (e.g., 'rainfall above 800mm', 'over-exploited blocks', 'Jaipur district analysis')"):
        # Add user message
        st.session_state.messages.append({"role": "user", "content": query})
        with st.chat_message("user"):
            st.markdown(query)

        # Perform search based on selected method
        with st.chat_message("assistant"):
            with st.spinner("Searching database..."):
                try:
                    # Check for district first to run targeted queries
                    detected_district = engine.detect_district_in_query(query)
                    # Check if this is a numeric parameter query
                    numeric_query = detect_numeric_query(query)

                    if numeric_query["is_numeric_query"]:
                        param = numeric_query["parameter"]
                        query_type = numeric_query["type"]
                        limit = int(numeric_query["limit"]) if numeric_query["limit"] else None

                        # For district-specific searches, get district blocks first, then filter
                        if detected_district:
                            district_variants = [detected_district, detected_district.upper(), detected_district.lower(), detected_district.title()]
                            district_blocks = []

                            for variant in district_variants:
                                district_blocks = engine.get_district_blocks(variant)
                                if district_blocks:
                                    break

                            if district_blocks:
                                # Apply numeric filtering to district blocks
                                filtered_results = []
                                for block in district_blocks:
                                    param_value = block.get(param, 0)

                                    if query_type == "above" and param_value >= numeric_query["min_value"]:
                                        filtered_results.append(block)
                                    elif query_type == "below" and param_value <= numeric_query["max_value"]:
                                        filtered_results.append(block)
                                    elif query_type == "between" and numeric_query["min_value"] <= param_value <= numeric_query["max_value"]:
                                        filtered_results.append(block)

                                # Sort results
                                if query_type in ["above", "between"] or query_type == "top_highest":
                                    filtered_results.sort(key=lambda x: x.get(param, 0), reverse=True)
                                else:
                                    filtered_results.sort(key=lambda x: x.get(param, 0))

                                # Limit results if specified
                                if limit:
                                    filtered_results = filtered_results[:limit]

                                results = filtered_results
                                search_type = f"district-filtered numeric"
                            else:
                                st.error(f"Could not find any blocks for district: {detected_district}")
                                results = []

                        else:
                            # Use global search methods for non-district queries
                            if query_type == "above":
                                results = engine.search_parameter_range(
                                    parameter=param,
                                    min_value=numeric_query["min_value"],
                                    top_k=limit,
                                    sort_order="desc"
                                )
                            elif query_type == "below":
                                results = engine.search_parameter_range(
                                    parameter=param,
                                    max_value=numeric_query["max_value"],
                                    top_k=limit,
                                    sort_order="asc"
                                )
                            elif query_type == "between":
                                results = engine.search_parameter_range(
                                    parameter=param,
                                    min_value=numeric_query["min_value"],
                                    max_value=numeric_query["max_value"],
                                    top_k=limit,
                                    sort_order="desc"
                                )
                            elif query_type in ["top_highest", "top_lowest"]:
                                highest = query_type == "top_highest"
                                results = engine.search_top_blocks_by_parameter(
                                    parameter=param,
                                    top_k=limit,
                                    highest=highest
                                )

                    else:
                        # Execute normal search
                        if detected_district:
                            # For text searches with district, get district blocks and do text matching
                            all_results = engine.search(query, top_k=50, min_score=0.0)
                            results = [r for r in all_results if r.get('district_name', '').upper() == detected_district.upper()][:20]
                        else:
                            # Normal search without district filtering
                            results = engine.search(query, top_k=20, min_score=0.1)

                    # Debug: Log search results count
                    #st.info(f"Found {len(results)} blocks matching criteria")
                    if not results:
                        answer = "No blocks found matching your criteria. Please try a different search or check if the district name is correct."
                    else:
                        # Format context and generate LLM response
                        context = format_context_for_llm(results, query)
                        prompt = generate_enhanced_prompt(context, query, numeric_query if numeric_query["is_numeric_query"] else None)

                        # Get LLM response
                        completion = client.chat.completions.create(
                            model="llama-3.1-8b-instant",
                            messages=[{"role": "user", "content": prompt}],
                            temperature=0.2,
                            max_tokens=1200,
                        )
                        answer = completion.choices[0].message.content

                    # Display the answer
                    st.markdown(answer)

                    # Display search results
                    display_search_results(results, "search")

                    # Add assistant message to chat history
                    st.session_state.messages.append({"role": "assistant", "content": answer})

                except Exception as e:
                    error_msg = f"Error during search: {str(e)}"
                    st.error(error_msg)

                    # Fallback: show raw results if available
                    try:
                        fallback_results = engine.search(query, top_k=5)
                        if fallback_results:
                            st.info("Here's some data that might help:")
                            display_search_results(fallback_results, "fallback")
                    except:
                        st.error("Unable to retrieve data. Please try again.")

if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token (sign up at https://dashboard.ngrok.com/get-started/your-authtoken)
ngrok.set_auth_token("2sZwdR1a0Y4a8OKLox4nQlyuVbC_oPTvp1uW6Di1AwMq2zb2")

# Open a tunnel to the streamlit port 8501
public_url = ngrok.connect(addr='8501')
print(f"Streamlit public URL: {public_url}")

# Run streamlit app in background
!streamlit run app.py &

Streamlit public URL: NgrokTunnel: "https://92a543421210.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.170.151:8501



2025-09-25 04:00:38.126373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758772838.148232    2150 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758772838.155194    2150 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758772838.172170    2150 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758772838.172197    2150 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758772838.172201    2150 computation_placer.cc:177] computation placer alr

In [ ]:
!pip install -U transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load IndicTrans2 translation model (Hindi to English)
model_name = "ai4bharat/indictrans2-indic-en-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True) # Add trust_remote_code=True
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True) # Add trust_remote_code=True

def translate_hi_to_en(text):
    # Format the input text with language tags
    formatted_text = f"hin_Deva eng_Latn {text}"
    inputs = tokenizer(formatted_text, return_tensors="pt")
    translated_tokens = model.generate(**inputs, max_length=100, use_cache=False)
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_text

# Example Hindi query
hindi_query = "मुझे भारत के बारे में जानकारी चाहिए।"

# Translate Hindi to English
english_query = translate_hi_to_en(hindi_query)
print("Original Hindi query:", hindi_query)
print("Translated English query:", english_query)

# You can now pass english_query to your RAG model inference pipeline

Original Hindi query: मुझे भारत के बारे में जानकारी चाहिए।
Translated English query: I need information about India.


In [ ]:
import pandas as pd
df = pd.read_csv('/content/rajasthan_groundwater_blocks_20250913_230226.csv')
freq = df['total_category'].value_counts()
print(freq)


total_category
over_exploited    240
safe               40
semi_critical      27
critical           25
salinity            3
Name: count, dtype: int64
